In [1]:
from march.pyt.mc import mc

import plotly.graph_objects as go
import torch
import time

In [2]:
def create_voxel_grid_loop(res_x, res_y, res_z, bounds, dtype=torch.float32, device='cpu'):
    """
    Create a voxel grid with vertices and cube indices.
    
    Args:
        res_x, res_y, res_z: Resolution (number of voxels) in each dimension
        bounds: Tuple of ((x_min, x_max), (y_min, y_max), (z_min, z_max))
        dtype: Data type for vertex coordinates (default: torch.float32)
    
    Returns:
        grids: Tensor of shape (num_vertices, 3) containing vertex coordinates
        cubes: Tensor of shape (num_cubes, 8) containing vertex indices for each cube
    """
    (x_min, x_max), (y_min, y_max), (z_min, z_max) = bounds
    
    # Create 1D coordinates for each dimension
    x_coords = torch.linspace(x_min, x_max, res_x + 1, dtype=dtype, device=device)
    y_coords = torch.linspace(y_min, y_max, res_y + 1, dtype=dtype, device=device)
    z_coords = torch.linspace(z_min, z_max, res_z + 1, dtype=dtype, device=device)
    
    # Create meshgrid: shape (res_x+1, res_y+1, res_z+1)
    zz, yy, xx = torch.meshgrid(z_coords, y_coords, x_coords, indexing='ij')
    
    # Flatten to get all vertices: shape (num_vertices, 3)
    grids = torch.stack([xx.flatten(), yy.flatten(), zz.flatten()], dim=1)
    
    # Create cube indices
    # For each cube at position (i, j, k), the 8 vertices follow the binary pattern:
    # (i+0, j+0, k+0), (i+1, j+0, k+0), (i+0, j+1, k+0), (i+1, j+1, k+0),
    # (i+0, j+0, k+1), (i+1, j+0, k+1), (i+0, j+1, k+1), (i+1, j+1, k+1)
    
    cube_indices = []
    for k in range(res_z):
        for j in range(res_y):
            for i in range(res_x):
                # Linear indices into the flattened grid
                stride_x = 1
                stride_y = res_x + 1
                stride_z = (res_y + 1) * (res_x + 1)
                
                v0 = k * stride_z + j * stride_y + i * stride_x
                v1 = k * stride_z + j * stride_y + (i + 1) * stride_x
                v2 = k * stride_z + (j + 1) * stride_y + i * stride_x
                v3 = k * stride_z + (j + 1) * stride_y + (i + 1) * stride_x
                v4 = (k + 1) * stride_z + j * stride_y + i * stride_x
                v5 = (k + 1) * stride_z + j * stride_y + (i + 1) * stride_x
                v6 = (k + 1) * stride_z + (j + 1) * stride_y + i * stride_x
                v7 = (k + 1) * stride_z + (j + 1) * stride_y + (i + 1) * stride_x
                
                cube_indices.append([v0, v1, v2, v3, v4, v5, v6, v7])
    
    cubes = torch.tensor(cube_indices, dtype=torch.long, device=device)
    
    return grids, cubes

In [3]:
def create_voxel_grid_torch(res_x, res_y, res_z, bounds, dtype=torch.float32, device='cpu'):
    """
    Create a voxel grid with vertices and cube indices (vectorized).
    
    Args:
        res_x, res_y, res_z: Resolution (number of voxels) in each dimension
        bounds: Tuple of ((x_min, x_max), (y_min, y_max), (z_min, z_max))
        dtype: Data type for vertex coordinates (default: torch.float32)
    
    Returns:
        grids: Tensor of shape (num_vertices, 3) containing vertex coordinates
        cubes: Tensor of shape (num_cubes, 8) containing vertex indices for each cube
    """
    (x_min, x_max), (y_min, y_max), (z_min, z_max) = bounds
    
    # Create 1D coordinates for each dimension
    x_coords = torch.linspace(x_min, x_max, res_x + 1, dtype=dtype, device=device)
    y_coords = torch.linspace(y_min, y_max, res_y + 1, dtype=dtype, device=device)
    z_coords = torch.linspace(z_min, z_max, res_z + 1, dtype=dtype, device=device)
    
    # Create meshgrid: shape (res_z+1, res_y+1, res_x+1)
    zz, yy, xx = torch.meshgrid(z_coords, y_coords, x_coords, indexing='ij')
    
    # Flatten to get all vertices: shape (num_vertices, 3)
    grids = torch.stack([xx.flatten(), yy.flatten(), zz.flatten()], dim=1)
    
    # Vectorized cube indices generation
    stride_x = 1
    stride_y = res_x + 1
    stride_z = (res_y + 1) * (res_x + 1)
    
    # Create all cube positions via meshgrid
    i_idx = torch.arange(res_x, device=device)
    j_idx = torch.arange(res_y, device=device)
    k_idx = torch.arange(res_z, device=device)

    k_grid, j_grid, i_grid = torch.meshgrid(k_idx, j_idx, i_idx, indexing='ij')
    
    # Compute base index for each cube
    base = k_grid * stride_z + j_grid * stride_y + i_grid * stride_x
    base = base.flatten().unsqueeze(1)  # shape (num_cubes, 1)
    
    # Define vertex offsets within a cube
    offsets = torch.tensor([
        [0, 0, 0],  # v0
        [1, 0, 0],  # v1
        [0, 1, 0],  # v2
        [1, 1, 0],  # v3
        [0, 0, 1],  # v4
        [1, 0, 1],  # v5
        [0, 1, 1],  # v6
        [1, 1, 1],  # v7
    ], dtype=torch.long, device=device)
    
    # Convert offsets to linear indices
    vertex_offsets = offsets[:, 0] * stride_x + offsets[:, 1] * stride_y + offsets[:, 2] * stride_z
    
    # Broadcast and add: shape (num_cubes, 8)
    cubes = base + vertex_offsets.unsqueeze(0)
    cubes = cubes.long()
    
    return grids, cubes

In [4]:
def create_voxel_grid_torch_fused(res_x, res_y, res_z, bounds, dtype=torch.float32, device='cpu'):
    """
    Create a voxel grid with vertices and cube indices (vectorized).
    
    Args:
        res_x, res_y, res_z: Resolution (number of voxels) in each dimension
        bounds: Tuple of ((x_min, x_max), (y_min, y_max), (z_min, z_max))
        dtype: Data type for vertex coordinates (default: torch.float32)
    
    Returns:
        grids: Tensor of shape (num_vertices, 3) containing vertex coordinates
        cubes: Tensor of shape (num_cubes, 8) containing vertex indices for each cube
    """
    (x_min, x_max), (y_min, y_max), (z_min, z_max) = bounds
    
    # Create 1D coordinates for each dimension
    x_coords = torch.linspace(x_min, x_max, res_x + 1, dtype=dtype, device=device)
    y_coords = torch.linspace(y_min, y_max, res_y + 1, dtype=dtype, device=device)
    z_coords = torch.linspace(z_min, z_max, res_z + 1, dtype=dtype, device=device)
    
    # Create meshgrid: shape (res_z+1, res_y+1, res_x+1)
    zz, yy, xx = torch.meshgrid(z_coords, y_coords, x_coords, indexing='ij')
    
    # Flatten to get all vertices: shape (num_vertices, 3)
    grids = torch.stack([xx.flatten(), yy.flatten(), zz.flatten()], dim=1)
    
    # Vectorized cube indices generation
    stride_x = 1
    stride_y = res_x + 1
    stride_z = (res_y + 1) * (res_x + 1)
    
    # Create all cube positions via meshgrid
    i_idx = torch.arange(res_x, device=device)
    j_idx = torch.arange(res_y, device=device)
    k_idx = torch.arange(res_z, device=device)

    k_grid, j_grid, i_grid = torch.meshgrid(k_idx, j_idx, i_idx, indexing='ij')
    
    # Define vertex offsets within a cube
    offsets = torch.tensor([
        [0, 0, 0],  # v0
        [1, 0, 0],  # v1
        [0, 1, 0],  # v2
        [1, 1, 0],  # v3
        [0, 0, 1],  # v4
        [1, 0, 1],  # v5
        [0, 1, 1],  # v6
        [1, 1, 1],  # v7
    ], dtype=torch.long, device=device)
    
    # Convert offsets to linear indices
    vertex_offsets = offsets[:, 0] * stride_x + offsets[:, 1] * stride_y + offsets[:, 2] * stride_z
    
    # Broadcast and add: shape (num_cubes, 8)
    cubes = (k_grid * stride_z + j_grid * stride_y + i_grid * stride_x).flatten().unsqueeze(1) + vertex_offsets.unsqueeze(0)
    cubes = cubes.long()
    
    return grids, cubes

In [5]:
res_x, res_y, res_z = 2, 1, 2

bounds = ((0, 2), (0, 1), (-1, 1))

time_start = time.time()
grids_loop, cubes_loop = create_voxel_grid_loop(res_x, res_y, res_z, bounds)

time_end = time.time()
print(f"Time taken: {time_end - time_start:.4f} seconds")

print("Grid Vertices:\n", grids_loop, "\nShape:", grids_loop.shape)
print("Cubes:\n", cubes_loop, "\nShape:", cubes_loop.shape)

Time taken: 0.0006 seconds
Grid Vertices:
 tensor([[ 0.,  0., -1.],
        [ 1.,  0., -1.],
        [ 2.,  0., -1.],
        [ 0.,  1., -1.],
        [ 1.,  1., -1.],
        [ 2.,  1., -1.],
        [ 0.,  0.,  0.],
        [ 1.,  0.,  0.],
        [ 2.,  0.,  0.],
        [ 0.,  1.,  0.],
        [ 1.,  1.,  0.],
        [ 2.,  1.,  0.],
        [ 0.,  0.,  1.],
        [ 1.,  0.,  1.],
        [ 2.,  0.,  1.],
        [ 0.,  1.,  1.],
        [ 1.,  1.,  1.],
        [ 2.,  1.,  1.]]) 
Shape: torch.Size([18, 3])
Cubes:
 tensor([[ 0,  1,  3,  4,  6,  7,  9, 10],
        [ 1,  2,  4,  5,  7,  8, 10, 11],
        [ 6,  7,  9, 10, 12, 13, 15, 16],
        [ 7,  8, 10, 11, 13, 14, 16, 17]]) 
Shape: torch.Size([4, 8])


In [6]:
time_start = time.time()
grids_torch, cubes_torch = create_voxel_grid_torch(res_x, res_y, res_z, bounds)

time_end = time.time()
print(f"Time taken: {time_end - time_start:.4f} seconds")

print("Grid Vertices:\n", grids_torch, "\nShape:", grids_torch.shape)
print("Cubes:\n", cubes_torch, "\nShape:", cubes_torch.shape)

Time taken: 0.0004 seconds
Grid Vertices:
 tensor([[ 0.,  0., -1.],
        [ 1.,  0., -1.],
        [ 2.,  0., -1.],
        [ 0.,  1., -1.],
        [ 1.,  1., -1.],
        [ 2.,  1., -1.],
        [ 0.,  0.,  0.],
        [ 1.,  0.,  0.],
        [ 2.,  0.,  0.],
        [ 0.,  1.,  0.],
        [ 1.,  1.,  0.],
        [ 2.,  1.,  0.],
        [ 0.,  0.,  1.],
        [ 1.,  0.,  1.],
        [ 2.,  0.,  1.],
        [ 0.,  1.,  1.],
        [ 1.,  1.,  1.],
        [ 2.,  1.,  1.]]) 
Shape: torch.Size([18, 3])
Cubes:
 tensor([[ 0,  1,  3,  4,  6,  7,  9, 10],
        [ 1,  2,  4,  5,  7,  8, 10, 11],
        [ 6,  7,  9, 10, 12, 13, 15, 16],
        [ 7,  8, 10, 11, 13, 14, 16, 17]]) 
Shape: torch.Size([4, 8])


In [7]:
print(torch.allclose(grids_loop, grids_torch))  # Should be True
print(torch.equal(cubes_loop, cubes_torch))  # Should be True

True
True


In [8]:
res_x, res_y, res_z = 128, 128, 128

time_start = time.time()
grids_loop, cubes_loop = create_voxel_grid_loop(res_x, res_y, res_z, ((-1, 1), (-1, 1), (-1, 1)), device='cpu')

time_end = time.time()
print(f"Time taken: {time_end - time_start:.4f} seconds")

time_start = time.time()
grids_torch, cubes_torch = create_voxel_grid_torch(res_x, res_y, res_z, ((-1, 1), (-1, 1), (-1, 1)), device='cpu')

time_end = time.time()

print(f"Time taken: {time_end - time_start:.4f} seconds")

torch.cuda.empty_cache()

Time taken: 2.2266 seconds
Time taken: 0.0132 seconds


In [9]:
res_x, res_y, res_z = 128, 128, 128

time_start = time.time()
grids_loop, cubes_loop = create_voxel_grid_loop(res_x, res_y, res_z, ((-1, 1), (-1, 1), (-1, 1)), device='cuda')

time_end = time.time()
print(f"Time taken: {time_end - time_start:.4f} seconds")

time_start = time.time()
grids_torch, cubes_torch = create_voxel_grid_torch(res_x, res_y, res_z, ((-1, 1), (-1, 1), (-1, 1)), device='cuda')

time_end = time.time()

print(f"Time taken: {time_end - time_start:.4f} seconds")

torch.cuda.empty_cache()

Time taken: 2.5835 seconds
Time taken: 0.0058 seconds


In [10]:
res_x, res_y, res_z = 1024, 1024, 1024

time_start = time.time()
grids_torch_fused, cubes_torch_fused = create_voxel_grid_torch_fused(res_x, res_y, res_z, ((-1, 1), (-1, 1), (-1, 1)), device='cuda')

time_end = time.time()

print(f"Time taken: {time_end - time_start:.4f} seconds")

torch.cuda.empty_cache()

OutOfMemoryError: CUDA out of memory. Tried to allocate 12.04 GiB. GPU 0 has a total capacity of 23.51 GiB of which 10.54 GiB is free. Including non-PyTorch memory, this process has 12.78 GiB memory in use. Of the allocated memory 12.33 GiB is allocated by PyTorch, and 10.82 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# iso = 0.0

# values = torch.tensor(
#     [
#         1,
#         -0.5,
#         1,
#         1,
#         -0.5,
#         1,
#         1,
#         -0.5,
#         1,
#         1,
#         -0.5,
#         1,
#         1,
#         1,
#         1,
#         1,
#         1,
#         1,
#         1
#     ]
# )

# verts, faces = mc(grids_loop, cubes_loop, values, iso)

# print("# verts:", verts.shape[0])
# print("# faces:", faces.shape[0])

# print("Verts:")
# print(verts)

# print("Faces:")
# print(faces)

In [ ]:
# # Create figure
# fig = go.Figure()

# # Create color array: red if value > iso, blue otherwise
# colors = ['red' if v > iso else 'blue' for v in values]

# # Add grid points as scatter plot
# fig.add_trace(go.Scatter3d(
#     x=grids_torch[:, 0],
#     y=grids_torch[:, 1],
#     z=grids_torch[:, 2],
#     mode='markers',
#     marker=dict(size=5, color=colors, opacity=0.8),
#     name='Grid Points'
# ))

# # Add mesh as wireframe
# # First, create the mesh surface with low opacit
# fig.add_trace(go.Mesh3d(
#     x=verts[:, 0],
#     y=verts[:, 1],
#     z=verts[:, 2],
#     i=faces[:, 0],
#     j=faces[:, 1],
#     k=faces[:, 2],
#     opacity=0.1,
#     color='lightblue',
#     showlegend=False
# ))

# # Add wireframe edges
# edges_x = []
# edges_y = []
# edges_z = []
# for i, j, k in faces:
#     # Edge 1: vertex i to j
#     edges_x.extend([verts[i, 0], verts[j, 0], None])
#     edges_y.extend([verts[i, 1], verts[j, 1], None])
#     edges_z.extend([verts[i, 2], verts[j, 2], None])
#     # Edge 2: vertex j to k
#     edges_x.extend([verts[j, 0], verts[k, 0], None])
#     edges_y.extend([verts[j, 1], verts[k, 1], None])
#     edges_z.extend([verts[j, 2], verts[k, 2], None])
#     # Edge 3: vertex k to i
#     edges_x.extend([verts[k, 0], verts[i, 0], None])
#     edges_y.extend([verts[k, 1], verts[i, 1], None])
#     edges_z.extend([verts[k, 2], verts[i, 2], None])

# fig.add_trace(go.Scatter3d(
#     x=edges_x,
#     y=edges_y,
#     z=edges_z,
#     mode='lines',
#     line=dict(color='darkblue', width=2),
#     name='Mesh Edges',
#     showlegend=True
# ))

# fig.update_layout(
#     scene=dict(
#         xaxis_title='X',
#         yaxis_title='Y',
#         zaxis_title='Z',
#         aspectmode='data'
#     ),
#     title='Grid Points and Marching Cubes Mesh (Wireframe)',
#     width=800,
#     height=800
# )

# fig.show()